In [193]:
import json
import os
import pandas as pd
import traceback
import langchain

In [194]:
from langchain_openai import ChatOpenAI

In [195]:
KEY = os.getenv("OPENAI_API_KEY")

In [196]:
from dotenv import load_dotenv
load_dotenv()   #take environment variables from .env


True

In [197]:
llm = ChatOpenAI(api_key=KEY, model_name="gpt-4.1-mini", temperature=0.5,)

In [198]:
llm

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x0000019DA3770730>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000019DA3771930>, root_client=<openai.OpenAI object at 0x0000019DA3771780>, root_async_client=<openai.AsyncOpenAI object at 0x0000019DA3770CD0>, model_name='gpt-4.1-mini', temperature=0.5, model_kwargs={}, openai_api_key=SecretStr('**********'))

In [199]:
from langchain.prompts import PromptTemplate

In [200]:
RESPONSE_JSON = {
    "1": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "2": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "3": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
}

In [201]:
TEMPLATE="""
Text:
{text}

You are an expert multiple-choice question (MCQ) creator. Based on the above text, your task is to create exactly {number} MCQs for {subject} students using a {tone} tone.

⚠️ Follow these rules strictly:
- Use ONLY the information in the text to generate the questions.
- Each question must have 4 answer choices.
- Clearly identify the correct answer.
- Do NOT repeat questions.
- Keep the language level suitable for {tone} tone.
- Format your output **exactly** like the RESPONSE_JSON below.

Your output MUST be a valid JSON object that exactly matches the structure of RESPONSE_JSON. No explanations, just the JSON.

### RESPONSE_JSON
{response_json}
"""

In [202]:
from langchain.chains import LLMChain
from langchain.chains import SequentialChain
from langchain.callbacks import get_openai_callback
import PyPDF2

In [203]:
from langchain.llms import OpenAI

In [204]:
quiz_generation_prompt = PromptTemplate(
    input_variables=["text", "number", "subject", "tone", "response_json"],
    template=TEMPLATE,
)

In [205]:
# New version of the LLMChain
from langchain_core.runnables import RunnableMap

In [206]:

# Define the new runnable chain
quiz_chain = (
    quiz_generation_prompt
    | llm
    | RunnableMap({"quiz": lambda x: x})
).with_config({"verbose": True})

In [219]:
TEMPLATE2 = """
You are an expert English grammarian and MCQ evaluator. You are given a Multiple Choice Quiz designed for {subject} students.

First, evaluate the complexity of the quiz and provide a short analysis (maximum 50 words) of whether the quiz matches the cognitive and analytical abilities of the students.

If the quiz is too difficult, too easy, grammatically incorrect, or unclear in any way, revise **only the necessary questions** to make the quiz appropriate for the target students.

🧠 Use the tone provided in the input: {tone}

⚠️ You MUST return your response in the following strict JSON format:

{
  "complexity_analysis": "short evaluation here",
  "updated_quiz": [
    {
      "question": "original or revised question",
      "options": ["option A", "option B", "option C", "option D"],
      "answer": "correct answer text"
    },
    ...
  ]
}

Rules:
- Copy unchanged questions exactly into `updated_quiz`.
- Do not add extra explanation or text outside the JSON.
- Ensure all questions are clear, grammatically correct, and aligned with the tone.
- Keep the number of questions exactly the same as the original input.

Quiz_MCQs:
{quiz}
"""


In [220]:
quiz_evaluation_prompt=PromptTemplate(input_variables=["subject", "quiz"], template=TEMPLATE2)

In [221]:
review_chain = quiz_evaluation_prompt | llm | RunnableMap({"review": lambda x: x}).with_config({"verbose": True})

In [222]:
from langchain_core.runnables import RunnableLambda

generate_evaluate_chain = (
    RunnableLambda(lambda x: x)  # Pass input through
    | RunnableLambda(lambda x: {"input": x, "quiz": llm.invoke(quiz_generation_prompt.format(**x))})
    | RunnableLambda(lambda x: {
        **x["input"],
        "quiz": x["quiz"]
    })
    | RunnableLambda(lambda x: (print(f"[DEBUG] Evaluation Input:\n{x}\n[DEBUG] Subject: {x.get('subject')}"), x)[1])
    | quiz_evaluation_prompt
    | llm
    | RunnableLambda(lambda x: {"review": x})
)

In [223]:
file_path = r"M:\mcq\data.txt"
file_path

'M:\\mcq\\data.txt'

In [224]:
with open(file_path, 'r',encoding='utf-8') as file:
    TEXT = file.read()

In [225]:
# Serialize the Python dictionary into a JSON-formatted string
json.dumps(RESPONSE_JSON)

'{"1": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "2": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "3": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}}'

# Manually giving the values

In [229]:
NUMBER=5 
SUBJECT="machine-learning"
TONE="simple"

In [228]:
#https://python.langchain.com/docs/modules/model_io/llms/token_usage_tracking

#How to setup Token Usage Tracking in LangChain
with get_openai_callback() as cb:
    response=generate_evaluate_chain.invoke(
        {
            "text": TEXT,
            "number": NUMBER,
            "subject":SUBJECT,
            "tone": TONE,
            "response_json": json.dumps(RESPONSE_JSON)
        }
        )

[DEBUG] Evaluation Input:
{'text': 'Machine learning is a method that allows computers to learn from data and improve their performance without being directly programmed. Instead of using fixed instructions, machines are trained with data, allowing them to recognize patterns and make decisions. For instance, by analyzing many images labeled as "cats" or "dogs," a computer can learn to classify new images on its own. There are three main types of machine learning: supervised learning, where the model is trained with labeled data; unsupervised learning, where the system finds patterns in unlabeled data; and reinforcement learning, where the machine learns through feedback in the form of rewards or penalties. Machine learning is widely used in applications such as recommendation systems, voice assistants, and spam detection. For effective learning, the quality, quantity, and clarity of data are essential. Though the field may seem complex, machine learning is fundamentally about enabling 

KeyError: 'Input to PromptTemplate is missing variables {\'\\n  "complexity_analysis"\'}.  Expected: [\'\\n  "complexity_analysis"\', \'quiz\', \'subject\', \'tone\'] Received: [\'text\', \'number\', \'subject\', \'tone\', \'response_json\', \'quiz\']\nNote: if you intended {\n  "complexity_analysis"} to be part of the string and not a variable, please escape it with double curly braces like: \'{{\n  "complexity_analysis"}}\'.\nFor troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/INVALID_PROMPT_INPUT '

In [216]:
print(f"Total Tokens:{cb.total_tokens}")
print(f"Prompt Tokens:{cb.prompt_tokens}")
print(f"Completion Tokens:{cb.completion_tokens}")
print(f"Total Cost:{cb.total_cost}")

Total Tokens:1665
Prompt Tokens:1212
Completion Tokens:453
Total Cost:0.0012096


In [217]:
response

{'review': AIMessage(content='The quiz appropriately matches the cognitive level of machine-learning students, focusing on fundamental concepts and distinctions. Questions require understanding rather than rote recall, promoting analytical thinking. However, Question 5’s phrasing is slightly vague and can be clarified for precision and fairness.\n\nRevised Question 5:  \n**Which of the following is NOT typically considered an application of machine learning?**  \na) Recommendation systems  \nb) Voice assistants  \nc) Spam detection  \nd) Operating system development', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 95, 'prompt_tokens': 735, 'total_tokens': 830, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_6f2eabb9a5',

In [218]:
output = response['review']
output

AIMessage(content='The quiz appropriately matches the cognitive level of machine-learning students, focusing on fundamental concepts and distinctions. Questions require understanding rather than rote recall, promoting analytical thinking. However, Question 5’s phrasing is slightly vague and can be clarified for precision and fairness.\n\nRevised Question 5:  \n**Which of the following is NOT typically considered an application of machine learning?**  \na) Recommendation systems  \nb) Voice assistants  \nc) Spam detection  \nd) Operating system development', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 95, 'prompt_tokens': 735, 'total_tokens': 830, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_6f2eabb9a5', 'id': 'cha